In [1]:
import json
import sys
from rich import print as rprint
from pathlib import Path
import re
from datetime import datetime


nb_dir = Path.cwd()

project_root = nb_dir.parent.parent

sys.path.insert(0, str(project_root))

In [2]:

db_books_file = Path(project_root / "data_reload/db_files/books.json")
db_b2p_file = Path(project_root / "data_reload/db_files/books2people.json")
db_admin_file = Path(project_root / "data_reload/db_files/book_admin.json")

with open(db_books_file, "r") as f:
   books = json.load(f)
with open(db_b2p_file, "r") as f:
   b2p = json.load(f)
with open(db_admin_file, "r") as f:
   admin = json.load(f)

# rprint(books[:2])
# rprint(b2p[:2])
books_dict = {i["composite_id"]: i for i in books}
# rprint(dict(list(books_dict.items())[:2]))
b2p_dict = {i["composite_id"]: i for i in b2p}
# rprint(dict(list(b2p_dict.items())[:2]))

missing_dict = {k: v for k, v in books_dict.items() if k not in b2p_dict}
rprint(dict(list(missing_dict.items())[:2]))
rprint(f"missing rows: {len(missing_dict)}")


{
    'symbolkunde_25_2_2': {
        'book_id': 26,
        'composite_id': 'symbolkunde_25_2_2',
        'is_active': 2,
        'is_removed': False,
        'title': 'Die Vergessene Bildersprache christlicher Kunst',
        'subtitle': 'Ein Führer zum Verständnis der Tier-, Engel- und Mariensymbolik',
        'publisher': 'Beck',
        'place_of_publication': 'München',
        'publication_year': 1995,
        'edition': 'Fünfte Auflage',
        'pages': 336,
        'format_original': None,
        'format_expanded': None,
        'condition': None,
        'copies': None,
        'illustrations': 'Mit 89 Abb.',
        'packaging': None,
        'topic_id': 5,
        'is_translation': False,
        'original_language': None,
        'is_multivolume': False,
        'series_title': "Beck'sche Reihe",
        'total_volumes': None,
        'created_at': '2026-05-22 10:36:31.631653',
        'updated_at': '2026-05-22 10:36:31.631653'
    },
    'symbolkunde_30_2_2': {
        'book_id': 31,
        'composite_id': 'symbolkunde_30_2_2',
        'is_active': 1,
        'is_removed': False,
        'title': 'ZEICHEN UND SYMBOLE',
        'subtitle': 'Ursprung, Geschichte, Bedeutung',
        'publisher': 'Könemann',
        'place_of_publication': None,
        'publication_year': 2005,
        'edition': None,
        'pages': 160,
        'format_original': 'OPbd.',
        'format_expanded': 'Originalpappband',
        'condition': 'Neu',
        'copies': 1,
        'illustrations': 'Bildband',
        'packaging': None,
        'topic_id': 5,
        'is_translation': False,
        'original_language': None,
        'is_multivolume': False,
        'series_title': None,
        'total_volumes': None,
        'created_at': '2026-05-22 10:36:31.631653',
        'updated_at': '2026-05-22 10:36:31.631653'
    }
}

missing rows: 2393

In [3]:
b2p_list_file = Path(project_root / "data_reload/fix_missing/b2p_list.json")
with open(b2p_list_file, "r") as f:
   b2p_list_check = json.load(f)

rprint(f"b2p_list entries: {len(b2p_list_check)}")


# rprint(b2p_list_check[:2])

b2p_cid_dict = {}
for v in b2p_list_check:
    b2p_cid_dict.setdefault(v["composite_id"], []).append(v)
# rprint(dict(list(b2p_cid_dict.items())[:2]))

rprint(f"books in dict from b2p list: {len(b2p_cid_dict)}")

b2p_ids = set(b2p_cid_dict)     # the 1892 composite_ids from the b2p list
book_ids = set(books_dict)      # all composite_ids in books

in_both = b2p_ids & book_ids        # ids in BOTH
only_in_b2p = b2p_ids - book_ids    # ids in the b2p list but NOT in books

# check_dict = {k: v for k, v in books_dict.items() if k not in b2p_dict_check}
# rprint(f"check rows: {len(check_dict)}")

rprint(f"in both: {len(in_both)}")
rprint(f"only in b2p: {len(only_in_b2p)}")
   # how many don't exist in books at all

missing_ids = set(missing_dict)     # the 2393 books with no b2p row

overlap = b2p_ids & missing_ids     # new-people books that are ALSO missing
rprint(f"overlap: {len(overlap)}")


b2p_list entries: 2390

books in dict from b2p list: 1892

in both: 1890

only in b2p: 2

overlap: 0

this hunch DIDN'T pan out, unfortunately, so next thing I did was to go hunt for information in the parsed data (earliest stage) to see whether the books had people data or what else could be wrong. 

In [4]:
parsed_data = {}
parsed_folder = Path(project_root / "data/parsed")
count = 0
for file in parsed_folder.iterdir():
    with open(file, "r") as f:
       entries = json.load(f)
    # rprint(entries[:1])
    # break

    for entry in entries:
        custom_id = entry["custom_id"]
        title = entry["parsed_entry"]["title"]
        subtitle = entry["parsed_entry"]["subtitle"]
        authors = entry["parsed_entry"]["authors"]
        editors = entry["parsed_entry"]["editors"]
        contributors = entry["parsed_entry"]["contributors"]
        translator = entry["parsed_entry"]["translator"]
        original_entry = entry["parsed_entry"]["administrative"]["original_entry"]
        corrected_by_api = entry["parsed_entry"]["administrative"]["corrected_by_api"]
        missing_person = entry["parsed_entry"]["administrative"]["missing_person"]
        multiple_editions = entry["parsed_entry"]["administrative"]["multiple_editions"]
        api_concerned = entry["parsed_entry"]["administrative"]["api_concerned"]
        problematic_multi_volume = entry["parsed_entry"]["administrative"]["problematic_multi_volume"]
        verification_notes = entry["parsed_entry"]["administrative"]["verification_notes"]
        # rprint(custom_id)
        parsed_data[custom_id] = {
            "title": title,
            "subtitle": subtitle,
            "authors": authors,
            "editors": editors,
            "contributors": contributors,
            "translator": translator,
            "original_entry": original_entry,
            "corrected_by_api": corrected_by_api,
            "missing_person": missing_person,
            "multiple_editions": multiple_editions,
            "api_concerned": api_concerned,
            "problematic_multi_volume": problematic_multi_volume,
            "verification_notes": verification_notes,
        }
        count += 1

# rprint(f"processed entries: {count}")
# rprint(f"entries in parsed_data: {len(parsed_data)}")

# with open("parsed_data_file.json", "w") as f:
#     json.dump(parsed_data, f, ensure_ascii=False, indent=2)


    # break



    # for c_id, entry in entries.items():
    #     rprint(c_id, entry)
    #     break

In [5]:
# missing_dict = {k: v for k, v in books_dict.items() if k not in b2p_dict}
# missing_info = {k: v for k, v in missing_dict.items() if k in parsed_data}
missing_info = {k: v for k, v in parsed_data.items() if k in missing_dict}

# rprint(f"text: {len(missing_info)}")

rprint(dict(list(missing_info.items())[:1]))
rprint(f"entries in missing with info: {len(missing_info)}")


missing_flag_count = 0
no_people_count = 0
empty_and_flagged = 0
no_people_parsed_data = {}
people_flagged_data = {}
missing_with_people = {}

for id, entry in missing_info.items():
    authors = entry["authors"]
    editors = entry["editors"]
    contributors = entry["contributors"]
    translator = entry["translator"]

    # a_count = len(authors)
    # e_count = len(editors)
    # c_count = len(contributors)
    has_people = (authors or editors or contributors or translator)
    if entry["missing_person"]:
        missing_flag_count +=1
    if not has_people:
        no_people_count += 1
        if entry["missing_person"]:
            empty_and_flagged += 1
    # if not has_people or (has_people and entry["missing_person"]):
    if not has_people:
        no_people_parsed_data[id] = entry
    elif entry["missing_person"]:
        people_flagged_data[id] = entry
    else:
        missing_with_people[id] = entry

# rprint(f"no people: {no_people_count}")
# rprint(f"missing: {missing_flag_count}")
# rprint(f"empty & flagged: {empty_and_flagged}")

# with open("no_people_parsed.json", "w") as f:
#     json.dump(no_people_parsed_data, f, ensure_ascii=False, indent=2)
# with open("people_flagged.json", "w") as f:
#     json.dump(people_flagged_data, f, ensure_ascii=False, indent=2)
# with open("missing_with_people.json", "w") as f:
#     json.dump(missing_with_people, f, ensure_ascii=False, indent=2)

rprint(len(missing_info))
rprint(f"no people total: {len(no_people_parsed_data)}")
rprint(f"flagged: {len(people_flagged_data)}")
rprint(f"other issues: {len(missing_with_people)}")


# 242


{
    'wien_200_9_10': {
        'title': 'XLVII. Jahresausstellung der Genossenschaft der Bildenden Künstler Wiens April-Juni 1926',
        'subtitle': None,
        'authors': [],
        'editors': [],
        'contributors': [],
        'translator': None,
        'original_entry': 'KÜNSTLERHAUS || XLVII. JAHRESAUSSTELLUNG || Der Genossenschaft der Bildenden Künstler 
Wiens April-Juni 1926. Künstlerhaus I. Karlsplatz 5. 70 S. Mit vielen Abb. OBrosch.',
        'corrected_by_api': False,
        'missing_person': False,
        'multiple_editions': False,
        'api_concerned': False,
        'problematic_multi_volume': False,
        'verification_notes': None
    }
}

entries in missing with info: 2393

2393

no people total: 298

flagged: 164

other issues: 1931

In [ ]:
matched_people_file = Path(project_root / "data/people/prepping4loading/people_matched.json")

with open(matched_people_file, "r") as f:
   matched_people = json.load(f)


comp_ids_found = {}

for person in matched_people.values():
   for entry in person["occurrences"]:
      comp_id = entry["composite_id"]
      comp_ids_found.setdefault(comp_id, []).append({
         "unified_id": person["unified_id"],
         **entry,
      })

missing_ids = set(missing_info)
in_file = missing_ids & set(comp_ids_found)

rprint("comp_ids_found:")
rprint(dict(list(comp_ids_found.items())[:1]))
rprint(f"in_file rows: {len(in_file)}")
rprint(f"in_file: {list(in_file)[:3]}")

rprint(f"still missing: {len(missing_ids - in_file)}")


comp_ids_found:

{
    'erstausgaben_1265_51_78': [
        {
            'unified_id': 'bartens_daniela',
            'display_name': 'BARTENS, Daniela',
            'composite_id': 'erstausgaben_1265_51_78',
            'sort_order': 1,
            'is_author': False,
            'is_editor': True,
            'is_contributor': False,
            'is_translator': False
        },
        {
            'unified_id': 'melzer_gerhard',
            'display_name': 'MELZER, Gerhard',
            'composite_id': 'erstausgaben_1265_51_78',
            'sort_order': 2,
            'is_author': False,
            'is_editor': True,
            'is_contributor': False,
            'is_translator': False
        }
    ]
}

in_file rows: 1337

in_file: ['literatur_81_4_12', 'briefe_39_2_15', 'biographie_286_12_27']

in file: 1337

still missing: 1056

In [7]:
book_db_file = Path(project_root / "data_reload/db_files/books.json")
people_db_file = Path(project_root / "data_reload/db_files/people.json")

with open(book_db_file, "r") as f:
   db_books = json.load(f)

with open(people_db_file, "r") as f:
   db_people = json.load(f)

book_id_dict = {v["composite_id"]: v["book_id"] for v in db_books}

person_id_dict = {v["unified_id"]: v["person_id"] for v in db_people}

rprint(dict(list(person_id_dict.items())[:3]))


{'gockel_gabriele': 1, 'allram_josef': 2, 'zedlitz_joseph_c': 3}

In [8]:
for uid, data in

SyntaxError: invalid syntax (525354464.py, line 1)